# DentalAI Pro: Oral Cancer Risk Assessment Training Pipeline
This notebook contains the complete code to download oral cancer datasets, train a **DenseNet121** classification network, evaluate performance, and generate Grad-CAM visual explainability maps.

## 1. Setup & Dataset Download
Downloads Oral Cancer / Oral Lesion datasets from Kaggle.

In [ ]:
!pip install -q kaggle

import os
from pathlib import Path

# Download dataset
!kaggle datasets download -d shivam17299/oral-cancer-lip-and-oral-cavity-datasets --unzip -p ../datasets/oral_cancer

## 2. Load Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import cv2

## 3. Data Preprocessing & Augmentation
Resize images to 224x224 and configure dataset loaders.

In [ ]:
img_size = (224, 224)
batch_size = 32
data_dir = "../datasets/oral_cancer"

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    zoom_range=0.2,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2],
    validation_split=0.3
)

train_generator = train_datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.5)

val_generator = val_datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    shuffle=False
)

test_generator = val_datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

## 4. Build DenseNet121 Classifier
We use DenseNet121 due to its dense connectivity blocks which excel at local feature identification in complex tissue textures.

In [ ]:
base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.4)(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.3)(x)
predictions = Dense(train_generator.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

## 5. Model Training

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
    ModelCheckpoint('../backend/models/cancer_model.h5', monitor='val_loss', save_best_only=True)
]

history = model.fit(
    train_generator,
    epochs=15,
    validation_data=val_generator,
    callbacks=callbacks
)

## 6. Fine-Tuning

In [ ]:
base_model.trainable = True
# Unfreeze last 40 layers
for layer in base_model.layers[:-40]:
    layer.trainable = False
    
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_fine = model.fit(
    train_generator,
    epochs=15,
    validation_data=val_generator,
    callbacks=callbacks
)

## 7. Performance Charts & ROC Curve

In [ ]:
Y_pred = model.predict(test_generator)
y_pred = np.argmax(Y_pred, axis=1)

print('Classification Report')
print(classification_report(test_generator.classes, y_pred, target_names=test_generator.class_indices.keys()))

# ROC Curve
plt.figure(figsize=(8, 6))
for i, class_name in enumerate(test_generator.class_indices.keys()):
    y_true_bin = (test_generator.classes == i).astype(int)
    fpr, tpr, _ = roc_curve(y_true_bin, Y_pred[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{class_name} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc='lower right')
plt.show()

## 8. Grad-CAM Visualization Demo
Here is the function to compute visual explanations on test images.

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]
    
    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

# To test, extract a sample test image
sample_batch = next(test_generator)
sample_img = sample_batch[0][0] # First image
img_array = np.expand_dims(sample_img, axis=0)

# Find last conv layer in DenseNet121 (typically 'conv5_block16_concat' or 'relu')
last_conv = 'conv5_block16_concat'

heatmap = make_gradcam_heatmap(img_array, model, last_conv)

# Display overlay
img_u8 = np.uint8(255 * sample_img)
heatmap_u8 = np.uint8(255 * heatmap)
heatmap_resized = cv2.resize(heatmap_u8, (img_u8.shape[1], img_u8.shape[0]))
heatmap_color = cv2.applyColorMap(heatmap_resized, cv2.COLORMAP_JET)
overlay = cv2.addWeighted(heatmap_color, 0.4, img_u8, 0.6, 0)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(sample_img)
plt.title("Original Mucosal Photo")

plt.subplot(1, 2, 2)
plt.imshow(overlay)
plt.title("Grad-CAM Heatmap Activation")
plt.show()